In [ ]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta

# 中文注释：支持用户在Jupyter中通过变量指定开始日期，也兼容命令行参数
import sys

# 中文注释：优先从Jupyter变量start_date_str获取开始日期，如果没有则尝试命令行参数，否则默认为当天0点
try:
    # 中文注释：Jupyter中可直接在上方单元格定义start_date_str = "2024-06-01"
    start_date_str = "2025-04-01"
except NameError:
    # 中文注释：如果Jupyter变量不存在，则尝试命令行参数
    if len(sys.argv) > 1:
        start_date_str = sys.argv[1]
    else:
        start_date_str = None

if start_date_str:
    try:
        # 中文注释：将用户输入的日期字符串转换为datetime对象
        start_time = datetime.strptime(start_date_str, "%Y-%m-%d")
    except ValueError:
        print("开始日期格式错误，请使用YYYY-MM-DD格式")
        exit()
else:
    # 中文注释：未指定则默认为当天0点
    now = datetime.now()
    start_time = datetime.strptime(now.strftime("%Y-%m-%d 00:00:00"), "%Y-%m-%d %H:%M:%S")

# 获取当前日期和时间
now = datetime.now()

query = f"""name:点击立即定制 |select phone,uid,count(0) "点击次数",date_format(max(__time__) ,'%Y-%m-%d %H:%i:%S') "最后点击时间" 
,date_format(min(__time__) ,'%Y-%m-%d %H:%i:%S') "首次点击时间" 
from log group by 1,2"""

click_uid_df = get_sls_data_by_query(
    query=query,
    from_time=start_time,
    to_time=now,
    project="xianmu-front-end-log",
    logstore="xm-mall",
)

if len(click_uid_df) <= 0:
    print("没有数据")
    exit()

click_uid_df = click_uid_df.drop(columns=["__source__", "__time__"], errors="ignore")

In [ ]:
# 获取token
import requests
import os

login_url = "https://admin.summerfarm.net/authentication/auth/username/login"
login_data = {
    "username": "peng.tang@summerfarm.net",
    "password": os.getenv("XIANMU_ADMIN_PASSWORD"),
}

token = requests.post(login_url, data=login_data).json()

token_str = token.get("data").get("token")

print(token)


headers = {
    "token": token_str,
    "xm-rqid": "create_fake_merchant_tp",
    "xm-uid": "2047",
    "Content-Type": "application/json;charset=UTF-8",
}

print(headers)

In [ ]:
import requests


def get_merchant_detail(uid=51637):
    uid = int(uid)

    url = "https://admin.summerfarm.net/sf-mall-manage/merchant/query/page"
    _headers = {
        "content-type": "application/json;charset=UTF-8",
        "token": headers["token"],
    }
    data = {"mId": uid, "pageIndex": 1, "pageSize": 10}
    response = requests.post(url, headers=_headers, json=data)
    return response.json().get("data", {}).get("list", [])[0]


click_uid_df["merchant"] = click_uid_df["uid"].apply(get_merchant_detail)
click_uid_df.head(4)

In [ ]:
click_uid_df = click_uid_df.sort_values(by="最后点击时间", ascending=False)

markdown_list = ["门店ID, 手机号, BD名字, 门店名称, 区域, 门店业态, 首次点击时间, 最后点击时间, 点击次数"]
for index, row in click_uid_df.iterrows():
    merchant = row["merchant"]
    phone = row["phone"]
    uid = row["uid"]
    adminRealName = merchant["adminRealName"]
    mname = merchant["mname"]
    areaName = merchant["areaName"]
    businessType = merchant.get("businessType",'无')
    firstClickTime = row["首次点击时间"]
    lastClickTime = row["最后点击时间"]
    clickTimes = row["点击次数"]
    markdown_list.append(
        f"{len(markdown_list)}. {uid}, {phone}, {adminRealName}, {mname}, {areaName}, {businessType}, {firstClickTime}, {lastClickTime}, {clickTimes}"
    )

markdown_output = "\n".join(markdown_list)
print(markdown_output)

In [ ]:
from datetime import datetime
import requests


url = (
    "https://open.feishu.cn/open-apis/bot/v2/hook/e7802cad-765c-44f8-ac45-186b28cd4c8a"
)


def send_feishu_notice_with_title_and_content(
    markdown_str: str,
    feishu_url=url,
    title="",
    error=False,
):
    feishu_message_obj = {
        "schema": "2.0",
        "header": {
            "template": "red" if error else "blue",
            "title": {
                "content": f"**{title}**",
                "tag": "lark_md",
            },
        },
        "body": {
            "elements": [
                {
                    "tag": "markdown",
                    "content": markdown_str,
                },
                {
                    "tag": "markdown",
                    "content": f"> 数据生成于:{datetime.now().strftime('%Y-%m-%d %H:%M')}\n> ",
                },
            ]
        },
    }
    headers = {"Content-Type": "application/json"}
    data = {"msg_type": "interactive", "card": feishu_message_obj}
    feishu_result = requests.post(
        url=feishu_url, json=data, headers=headers, verify=False, proxies={}
    ).json()
    return feishu_result

date_of_today = datetime.now().strftime("%Y-%m-%d")
send_feishu_notice_with_title_and_content(markdown_str=markdown_output, title=f"点击立即定制的门店列表_{start_date_str}迄今")